# Placement Prediction 🚀

## 1. Load Trained Model and Preprocessor

In this section, we load the fitted preprocessing pipeline and the selected machine learning model from the `models` directory.
These components will be used to transform new student data and generate placement predictions.

In [1]:
import pandas as pd
import joblib

In [2]:
preprocessor = joblib.load("../models/preprocessor.pkl")

In [3]:
best_model = joblib.load("../models/best_model.pkl")

In [4]:
type(preprocessor), type(best_model)

(sklearn.compose._column_transformer.ColumnTransformer,
 sklearn.ensemble._gb.GradientBoostingClassifier)

In [8]:
df = pd.read_csv("../data/processed/student_placement_features.csv")

In [9]:
df.columns.tolist()

['branch',
 'college_tier',
 'cgpa',
 'backlogs',
 'coding_skills',
 'dsa_score',
 'aptitude_score',
 'communication_skills',
 'ml_knowledge',
 'system_design',
 'internships',
 'projects_count',
 'certifications',
 'hackathons',
 'open_source_contributions',
 'extracurriculars',
 'placement_status',
 'technical_skill_score',
 'has_backlog',
 'experience_score',
 'technical_skill_gap']

## 2. Create New Student Input

A sample student profile is created using the original features collected from a student.

The engineered features will be calculated automatically using the same feature-engineering logic used during model development.

In [10]:
student_data = {
    "branch": "CSE",
    "college_tier": "Tier-1",
    "cgpa": 8.5,
    "backlogs": 0,
    "coding_skills": 8,
    "dsa_score": 8,
    "aptitude_score": 8,
    "communication_skills": 8,
    "ml_knowledge": 7,
    "system_design": 6,
    "internships": 2,
    "projects_count": 3,
    "certifications": 3,
    "hackathons": 2,
    "open_source_contributions": 2,
    "extracurriculars": 5
}

In [11]:
student_df = pd.DataFrame([student_data])
student_df

,branch,college_tier,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,internships,projects_count,certifications,hackathons,open_source_contributions,extracurriculars
0,CSE,Tier-1,8.5,0,8,8,8,8,7,6,2,3,3,2,2,5


## 3. Apply Feature Engineering

The new student's original features are transformed using the same feature-engineering logic used during model development.

This ensures that the new input has the same feature structure as the data used to train the placement prediction model.

In [12]:
student_df["technical_skill_score"] = student_df[
    ["coding_skills", "dsa_score", "ml_knowledge", "system_design"]
].mean(axis=1)

In [13]:
student_df["experience_score"] = student_df[
    [
        "internships",
        "projects_count",
        "certifications",
        "hackathons",
        "open_source_contributions",
        "extracurriculars"
    ]
].sum(axis=1)

In [14]:
student_df["has_backlog"] = (
    student_df["backlogs"] > 0
).astype(int)

In [15]:
maximum_skill = student_df[
    ["coding_skills", "dsa_score", "ml_knowledge", "system_design"]
].max(axis=1)

minimum_skill = student_df[
    ["coding_skills", "dsa_score", "ml_knowledge", "system_design"]
].min(axis=1)

student_df["technical_skill_gap"] = maximum_skill - minimum_skill

In [16]:
student_df

,branch,college_tier,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,internships,projects_count,certifications,hackathons,open_source_contributions,extracurriculars,technical_skill_score,experience_score,has_backlog,technical_skill_gap
0,CSE,Tier-1,8.5,0,8,8,8,8,7,6,2,3,3,2,2,5,7.25,17,0,2


## 4. Prepare Features for Prediction

The engineered student profile is prepared in the same feature structure used during model training.

The fitted preprocessing pipeline is then used to transform the categorical and numerical features into the format expected by the trained model.

In [18]:
student_encoded = preprocessor.transform(student_df)
student_encoded.shape

(1, 28)

## 5. Generate Placement Prediction

The processed student profile is passed to the trained Gradient Boosting model to predict the student's placement status.

The model returns:
- `1` → Placed
- `0` → Not Placed

In [20]:
prediction = best_model.predict(student_encoded)
prediction

array([1])

## 6. Convert Prediction to Placement Status

The numerical prediction returned by the model is converted into a human-readable placement status.

In [21]:
placement_status = "Placed" if prediction[0] == 1 else "Not Placed"
placement_status

'Placed'

## 7. Prediction Probability

The trained model's prediction probability is used to understand the model's confidence in the predicted placement status.

In [22]:
prediction_probability = best_model.predict_proba(student_encoded)
prediction_probability

array([[0.1137526, 0.8862474]])

## 8. Final Placement Prediction

The predicted placement status and its corresponding probability are combined into a user-friendly result.

In [23]:
placement_probability = prediction_probability[0][1] * 100
placement_probability

np.float64(88.62474035147827)

In [24]:
placement_probability = round(placement_probability, 2)
placement_probability

np.float64(88.62)

In [26]:
result = {
    "placement_status": placement_status,
    "placement_probability": placement_probability
}
result

{'placement_status': 'Placed', 'placement_probability': np.float64(88.62)}